# WDC JSON-LD Retrieval for Fact-Checking Domains

Retrieves JSON-LD structured data from the **Web Data Commons (WDC) October 2024** release
for a set of fact-checking and news domains (Wikipedia, PolitiFact, CNN, Le Monde, Al Jazeera, …)
**without downloading the full ~750 GB dataset to disk**.

This notebook runs in parallel with the Common Crawl WARC pipeline (`extract_jsonld_commoncrawl.ipynb`).

## Strategy

**Part 1 — Coverage check (WDC stats pages)**
Scrape the WDC HTML ranking pages (top 1,000 domains by triples per format) to confirm
which target domains are present and at what scale. No data download needed.

**Part 2 — Streaming nquads retrieval**
WDC 2024-12 publishes ~7,462 nquad shards (~100 MB compressed each, ~751 GB total).
These shards are not sorted or indexed by domain, so we stream all of them through
memory, filtering line-by-line for our target PLDs. **Only matched triples are kept on disk.**
The full 751 GB passes through the network but never touches local storage.

> **Bandwidth note:** WDC is university-hosted and rate-limits aggressively.
> Keep `MAX_WORKERS` low (2) and the built-in per-request delay in place,
> or you'll get 429s and connection refusals within the first few dozen shards.

> **PLD granularity:** WDC indexes at pay-level domain level (`wikipedia.org`), so
> `en.wikipedia.org` appears as `wikipedia.org` in the stats and data.

> **Note:** WDC nquads contain one triple per line. This differs from the CC pipeline,
> which extracts the full JSON-LD blob per page. Both outputs are complementary:
> nquads are easier for triple-level analysis; blobs preserve the original annotation structure.

In [1]:
# ── Dependencies ───────────────────────────────────────────────────────────────
# Uncomment and run once if needed:
# !pip install requests beautifulsoup4 pandas lxml

In [2]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import requests
import pandas as pd
import time
import re
import gzip
from collections import defaultdict
from urllib.parse import urlparse
from bs4 import BeautifulSoup

TARGET_DOMAINS = [
    # Knowledge bases
    "en.wikipedia.org",
    # Dedicated fact-checkers
    "politifact.com",
    "snopes.com",
    "factcheck.org",
    "fullfact.org",
    "africacheck.org",
    # News wire services
    "reuters.com",
    "apnews.com",
    # International broadcasters
    "bbc.com",
    "aljazeera.com",
    "france24.com",
    "dw.com",
    # Major newspapers
    "theguardian.com",
    "nytimes.com",
    "thetimes.com",
    "lemonde.fr",
    "cnn.com",
]

# Pay-level domain (eTLD+1) — what WDC indexes on
# e.g. en.wikipedia.org → wikipedia.org
def get_pld(host):
    parts = host.split(".")
    return ".".join(parts[-2:]) if len(parts) >= 2 else host

pld_to_targets = defaultdict(list)
for d in TARGET_DOMAINS:
    pld_to_targets[get_pld(d)].append(d)

TARGET_PLDS = set(pld_to_targets.keys())

print(f"Target domains  : {len(TARGET_DOMAINS)}")
print(f"Target PLDs     : {len(TARGET_PLDS)}")
print("\nPLD → subdomain(s):")
for pld, hosts in sorted(pld_to_targets.items()):
    print(f"  {pld:<25} ← {hosts}")

Target domains  : 17
Target PLDs     : 17

PLD → subdomain(s):
  africacheck.org           ← ['africacheck.org']
  aljazeera.com             ← ['aljazeera.com']
  apnews.com                ← ['apnews.com']
  bbc.com                   ← ['bbc.com']
  cnn.com                   ← ['cnn.com']
  dw.com                    ← ['dw.com']
  factcheck.org             ← ['factcheck.org']
  france24.com              ← ['france24.com']
  fullfact.org              ← ['fullfact.org']
  lemonde.fr                ← ['lemonde.fr']
  nytimes.com               ← ['nytimes.com']
  politifact.com            ← ['politifact.com']
  reuters.com               ← ['reuters.com']
  snopes.com                ← ['snopes.com']
  theguardian.com           ← ['theguardian.com']
  thetimes.com              ← ['thetimes.com']
  wikipedia.org             ← ['en.wikipedia.org']


## Part 1 — WDC HTML Stats Pages (Top 1,000 domains per format)

WDC publishes lightweight HTML pages with the top 1,000 domains by extracted triples for each format.  
We scrape them all and search for our target domains.  
No authentication, no huge downloads — each page is a small HTML table.

In [3]:
# ── WDC stats page URLs ────────────────────────────────────────────────────────
WDC_STATS_BASE = "http://webdatacommons.org/structureddata/2024-12/stats/"

# Each format has a "More" page with top 1,000 domains by extracted triples.
FORMAT_PAGES = {
    "JSON-LD":      "top_domains_by_extracted_triples_for_extractor_html-embedded-jsonld.html",
    "Microdata":    "top_domains_by_extracted_triples_for_extractor_html-microdata.html",
    "RDFa":         "top_domains_by_extracted_triples_for_extractor_html-rdfa.html",
    "MF-hCard":     "top_domains_by_extracted_triples_for_extractor_html-mf-hcard.html",
    "MF-XFN":       "top_domains_by_extracted_triples_for_extractor_html-mf-xfn.html",
    "MF-adr":       "top_domains_by_extracted_triples_for_extractor_html-mf-adr.html",
    "MF-geo":       "top_domains_by_extracted_triples_for_extractor_html-mf-geo.html",
    "MF-hCalendar": "top_domains_by_extracted_triples_for_extractor_html-mf-hcalendar.html",
    "MF-hReview":   "top_domains_by_extracted_triples_for_extractor_html-mf-hreview.html",
    "MF-hListing":  "top_domains_by_extracted_triples_for_extractor_html-mf-hlisting.html",
    "MF-hRecipe":   "top_domains_by_extracted_triples_for_extractor_html-mf-hrecipe.html",
    "MF-hResume":   "top_domains_by_extracted_triples_for_extractor_html-mf-hresume.html",
    "MF-species":   "top_domains_by_extracted_triples_for_extractor_html-mf-species.html",
}


def scrape_domain_table(url):
    """
    Fetch a WDC stats HTML page and extract the domain ranking list.

    The pages do NOT use <table> — they use a numbered list of anchor tags
    with triple counts in parentheses, e.g.:
        <li><a href="...">blogspot.com</a> (30,926,064)</li>
    or inline text like:
        1. blogspot.com (30,926,064)

    Strategy:
    1. Try <table> (fallback for older pages).
    2. Try <ol>/<ul> list items containing <a> tags.
    3. Regex over the raw HTML for the pattern: domain + (number).
    Returns a DataFrame with columns: rank, domain, triples.
    """
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
    except Exception as e:
        print(f"  ERROR fetching {url}: {e}")
        return pd.DataFrame()

    html  = resp.text
    soup  = BeautifulSoup(html, "lxml")
    rows  = []

    # ── Strategy 1: <table> ────────────────────────────────────────────────────
    table = soup.find("table")
    if table:
        for tr in table.find_all("tr"):
            cells = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
            if cells:
                rows.append(cells)
        if len(rows) > 1:
            header = rows[0]
            data   = rows[1:]
            df = pd.DataFrame(data, columns=header[:len(data[0])] if data else header)
            col_lower = {c: c.lower() for c in df.columns}
            domain_col = next((c for c, cl in col_lower.items() if any(k in cl for k in ('domain','host','pld'))), df.columns[0])
            triple_col = next((c for c, cl in col_lower.items() if any(k in cl for k in ('triple','count'))), df.columns[1] if len(df.columns) > 1 else None)
            df = df.rename(columns={domain_col: "domain"})
            if triple_col:
                df = df.rename(columns={triple_col: "triples"})
                df["triples"] = pd.to_numeric(df["triples"].astype(str).str.replace(r"[,\s]", "", regex=True), errors="coerce")
            df.insert(0, "rank", range(1, len(df) + 1))
            df["domain"] = df["domain"].astype(str).str.strip()
            return df

    # ── Strategy 2: <li> items with <a> links ─────────────────────────────────
    # Pattern: <a href="...">domain.tld</a> followed by (N,NNN,NNN)
    NUM_RE = re.compile(r'\(\s*([\d,]+)\s*\)')
    list_items = soup.find_all("li")
    for item in list_items:
        a = item.find("a")
        if a:
            domain_text = a.get_text(strip=True)
            rest = item.get_text(separator=" ")
            m = NUM_RE.search(rest)
            triples = int(m.group(1).replace(",", "")) if m else None
            rows.append([domain_text, triples])

    if rows:
        df = pd.DataFrame(rows, columns=["domain", "triples"])
        df["domain"]  = df["domain"].astype(str).str.strip()
        df["triples"] = pd.to_numeric(df["triples"], errors="coerce")
        df.insert(0, "rank", range(1, len(df) + 1))
        return df

    # ── Strategy 3: regex over raw HTML ───────────────────────────────────────
    # Matches lines like: blogspot.com (30,926,064)  or  1. bbc.com (123,456)
    INLINE_RE = re.compile(
        r'(?:^\s*\d+\.\s+)?'           # optional leading rank "1. "
        r'([\w][\w\-]*(?:\.[\w\-]+)+)' # domain (at least one dot)
        r'\s*\(\s*([\d,]+)\s*\)',       # (triple_count)
        re.MULTILINE
    )
    for i, m in enumerate(INLINE_RE.finditer(html), start=1):
        domain  = m.group(1).strip()
        triples = int(m.group(2).replace(",", ""))
        rows.append([i, domain, triples])

    if rows:
        df = pd.DataFrame(rows, columns=["rank", "domain", "triples"])
        df["domain"] = df["domain"].astype(str).str.strip()
        return df

    print(f"  Could not parse any domain rows from {url}")
    return pd.DataFrame()


# ── Fetch all format pages and filter for target domains ───────────────────────
results = []

for fmt_name, page_file in FORMAT_PAGES.items():
    url = WDC_STATS_BASE + page_file
    print(f"Fetching {fmt_name} ... ", end="", flush=True)
    df = scrape_domain_table(url)
    if df.empty:
        print("(empty)")
        continue
    print(f"{len(df)} rows")

    matched = df[df["domain"].isin(TARGET_PLDS)].copy()
    if not matched.empty:
        matched.insert(0, "format", fmt_name)
        results.append(matched)
    time.sleep(0.3)


Fetching JSON-LD ... 1000 rows
Fetching Microdata ... 1000 rows
Fetching RDFa ... 1000 rows
Fetching MF-hCard ... 1000 rows
Fetching MF-XFN ... 1000 rows
Fetching MF-adr ... 1000 rows
Fetching MF-geo ... 1000 rows
Fetching MF-hCalendar ... 1000 rows
Fetching MF-hReview ... 1000 rows
Fetching MF-hListing ... 1000 rows
Fetching MF-hRecipe ... 1000 rows
Fetching MF-hResume ... 61 rows
Fetching MF-species ... 291 rows


In [4]:
# ── Display results ────────────────────────────────────────────────────────────
if results:
    combined = pd.concat(results, ignore_index=True)

    print("\n" + "="*70)
    print("FOUND — Target domains in WDC October 2024 (top 1,000 per format)")
    print("="*70)
    # Show all columns available
    print(combined.to_string(index=False))

    print("\n" + "="*70)
    print("JSON-LD only:")
    print("="*70)
    jsonld = combined[combined["format"] == "JSON-LD"]
    print(jsonld.to_string(index=False) if not jsonld.empty else "  None found in top 1,000")
else:
    print("No matches found in any format's top 1,000.")

# Report which domains were NOT found at all
if results:
    found_plds = set(combined["domain"].unique())
else:
    found_plds = set()

missing_plds = TARGET_PLDS - found_plds
if missing_plds:
    print("\n" + "="*70)
    print("NOT in top 1,000 for any format:")
    for pld in sorted(missing_plds):
        print(f"  {pld}  (requested: {pld_to_targets[pld]})")
    print("→ These may still be in the dataset but below the top-1,000 threshold.")
    print("  Use Part 2 (CC CDX API) and Part 3 (streaming) to investigate further.")


FOUND — Target domains in WDC October 2024 (top 1,000 per format)
      format  rank         domain  triples
     JSON-LD     5  wikipedia.org      NaN
     JSON-LD   158        cnn.com      NaN
    MF-hCard     2  wikipedia.org      NaN
      MF-adr     6  wikipedia.org      NaN
      MF-adr    14 politifact.com      NaN
      MF-geo     1  wikipedia.org      NaN
MF-hCalendar     1  wikipedia.org      NaN
  MF-hRecipe    49  wikipedia.org      NaN
  MF-species     1  wikipedia.org      NaN

JSON-LD only:
 format  rank        domain  triples
JSON-LD     5 wikipedia.org      NaN
JSON-LD   158       cnn.com      NaN

NOT in top 1,000 for any format:
  africacheck.org  (requested: ['africacheck.org'])
  aljazeera.com  (requested: ['aljazeera.com'])
  apnews.com  (requested: ['apnews.com'])
  bbc.com  (requested: ['bbc.com'])
  dw.com  (requested: ['dw.com'])
  factcheck.org  (requested: ['factcheck.org'])
  france24.com  (requested: ['france24.com'])
  fullfact.org  (requested: ['fullfac

## Part 2 — WDC Nquads: Retrieve Triples for Target Domains

WDC publishes structured data as nquads (one triple per line), sharded across
~7,462 gzipped files (~751 GB total for 2024-12 JSON-LD). Shards are not sorted
or indexed by domain, so there's no shortcut — we stream every shard, filter
line-by-line, and keep only matching triples.

**Pipeline:**
1. Fetch the shard URL list (~a few hundred KB)
2. Stream each shard through a gzip decoder, filtering for target PLDs inline
3. Append matched nquads to `wdc_jsonld_triples.nq`
4. Summarize by domain and `@type` in the final cell

**Expect multi-hour runtime.** Nothing is written to disk except matched lines,
so output size scales with match count (likely tens of MB to a few GB, not 751 GB).

In [5]:
import gzip, time

WDC_FILES_BASE = "http://webdatacommons.org/structureddata/2024-12/files/"
EXTRACTOR      = "html-embedded-jsonld"

RELEASES = ["2024-12", "2023-12", "2022-12"]

shard_urls     = []
active_release = None

for release in RELEASES:
    list_url = f"http://webdatacommons.org/structureddata/{release}/files/{EXTRACTOR}.list"
    try:
        resp = requests.get(list_url, timeout=20)
        resp.raise_for_status()
        candidate_urls = [l.strip() for l in resp.text.strip().splitlines() if l.strip()]
        if not candidate_urls:
            continue
        with requests.get(candidate_urls[0], stream=True, timeout=30) as r:
            r.raise_for_status()
            next(r.iter_content(chunk_size=32), b"")
        print(f"  {release}: OK ({len(candidate_urls):,} shards)")
        shard_urls     = candidate_urls
        active_release = release
        break
    except Exception as e:
        print(f"  {release}: {e}")

if not shard_urls:
    raise RuntimeError("All WDC releases unreachable. Try again later.")

print(f"\nUsing release : {active_release}")
print(f"Total shards  : {len(shard_urls):,}")

  2024-12: OK (7,462 shards)

Using release : 2024-12
Total shards  : 7,462


In [6]:
# WDC 2024-12 shards are not sorted or indexed by domain, so we must scan all of them.
# Filtering happens line-by-line inside the streaming cell below; bandwidth is the cost,
# not disk — only matched lines are kept.
all_shard_indices = list(range(len(shard_urls)))
print(f"Shards to scan: {len(all_shard_indices):,}")

Shards to scan: 7,462


In [ ]:
import threading, random
from concurrent.futures import ThreadPoolExecutor, as_completed

WDC_OUTPUT     = "wdc_jsonld_triples.nq"
TARGET_PATTERN = re.compile("|".join(re.escape(d) for d in TARGET_PLDS))

# Politeness knobs. WDC is university-hosted and will rate-limit (429) or refuse
# connections if you hit it hard. Keep workers low and add jittered delays.
MAX_WORKERS    = 2        # empirically, 4+ triggers 429 quickly
BASE_DELAY     = 1.0      # seconds between starting requests, per worker
JITTER         = 0.5      # +/- seconds random jitter
MAX_RETRIES    = 3
REQUEST_TIMEOUT = 600     # some shards are slow; don't give up too early


def stream_shard(shard_idx: int):
    """Stream one shard, filter lines by target PLD, return (matched_lines, total_lines).

    Retries on 429/503 and transient network errors with exponential backoff.
    Honors Retry-After when present. On final failure, returns ([], 0) and logs.
    """
    url = shard_urls[shard_idx]

    # Jittered pre-request delay to spread out workers
    time.sleep(BASE_DELAY + random.uniform(-JITTER, JITTER))

    for attempt in range(MAX_RETRIES):
        matched = []
        total   = 0
        try:
            with requests.get(url, stream=True, timeout=REQUEST_TIMEOUT) as r:
                if r.status_code in (429, 503):
                    # Respect server backpressure
                    retry_after = r.headers.get("Retry-After")
                    wait = int(retry_after) if (retry_after and retry_after.isdigit()) else (2 ** attempt) * 5
                    print(f"  shard {shard_idx}: {r.status_code}, sleeping {wait}s (attempt {attempt+1}/{MAX_RETRIES})")
                    time.sleep(wait)
                    continue
                r.raise_for_status()
                with gzip.GzipFile(fileobj=r.raw) as gz:
                    for raw in gz:
                        total += 1
                        line     = raw.decode("utf-8", errors="replace")
                        subj_end = line.find(" ")
                        subject  = line[:subj_end] if subj_end > 0 else line
                        if TARGET_PATTERN.search(subject):
                            matched.append(line)
                return matched, total
        except requests.exceptions.RequestException as e:
            wait = (2 ** attempt) * 5   # 5, 10, 20, 40, 80s
            print(f"  shard {shard_idx}: {type(e).__name__}, sleeping {wait}s (attempt {attempt+1}/{MAX_RETRIES})")
            time.sleep(wait)

    print(f"  shard {shard_idx}: GIVING UP after {MAX_RETRIES} attempts")
    return [], 0


stats      = {"done": 0, "failed": 0, "lines_total": 0, "lines_matched": 0}
stats_lock = threading.Lock()
file_lock  = threading.Lock()

open(WDC_OUTPUT, "w").close()

print(f"Streaming {len(all_shard_indices):,} shard(s) with {MAX_WORKERS} workers")
print(f"Politeness: {BASE_DELAY}s base delay +/- {JITTER}s jitter, up to {MAX_RETRIES} retries per shard\n")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(stream_shard, i): i for i in all_shard_indices}
    for future in as_completed(futures):
        idx = futures[future]
        matched_lines, total = future.result()

        if matched_lines:
            with file_lock:
                with open(WDC_OUTPUT, "a") as f:
                    f.writelines(matched_lines)

        with stats_lock:
            stats["done"]          += 1
            stats["lines_total"]   += total
            stats["lines_matched"] += len(matched_lines)
            if total == 0:
                stats["failed"] += 1
            # Print progress when matches found OR every 100 shards
            if len(matched_lines) > 0 or stats["done"] % 100 == 0:
                print(f"  [{stats['done']:>4}/{len(all_shard_indices):,}] shard {idx:05d}: "
                      f"{len(matched_lines):>7,} matched / {total:>9,} total  "
                      f"(running: {stats['lines_matched']:,} matched, {stats['failed']} failed)")

print(f"\nDone.")
print(f"  Shards processed : {stats['done']:,}")
print(f"  Shards failed    : {stats['failed']:,}")
print(f"  Lines scanned    : {stats['lines_total']:,}")
print(f"  Lines matched    : {stats['lines_matched']:,}")
print(f"  Output           : {WDC_OUTPUT}")

Streaming 7,462 shard(s) with 1 workers
Politeness: 1.0s base delay +/- 0.5s jitter, up to 3 retries per shard



In [ ]:
from collections import Counter, defaultdict

# nquads line format: <subject> <predicate> <object> <graph> .
NQ_RE = re.compile(r'^(<[^>]+>)\s+(<[^>]+>)\s+(.*?)\s+(<[^>]+>)\s+\.$')

RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"

domain_stats   = defaultdict(lambda: {"pages": set(), "triples": 0, "types": Counter()})
parse_errors   = 0

with open(WDC_OUTPUT) as f:
    for line in f:
        line = line.strip()
        m = NQ_RE.match(line)
        if not m:
            parse_errors += 1
            continue
        subject, predicate, obj, graph = m.groups()

        # Map subject to a target PLD
        matched_pld = next((p for p in TARGET_PLDS if p in subject), None)
        if not matched_pld:
            continue

        ds = domain_stats[matched_pld]
        ds["pages"].add(subject)
        ds["triples"] += 1

        if predicate == RDF_TYPE:
            # Extract local name: <http://schema.org/NewsArticle> → NewsArticle
            type_name = obj.strip("<>").split("/")[-1].split("#")[-1]
            ds["types"][type_name] += 1

print(f"{'='*65}")
print(f"WDC JSON-LD triples — summary by domain")
print(f"{'='*65}\n")
for domain in sorted(domain_stats):
    ds = domain_stats[domain]
    print(f"{domain}")
    print(f"  Unique pages  : {len(ds['pages']):>10,}")
    print(f"  Total triples : {ds['triples']:>10,}")
    if ds["types"]:
        print(f"  Top @types    :")
        for t, c in ds["types"].most_common(8):
            print(f"    {t:<40} {c:>8,}")
    print()

if parse_errors:
    print(f"(parse errors skipped: {parse_errors:,})")